# 🥇 Notebook 03 — Gold Features

**Project:** Energy Fraud & Default Risk Detection  
**Layer:** Gold (Silver → Aggregated Features → ML-Ready)  
**Author:** Zara Louise  
**Stack:** PySpark + Delta Lake + Unity Catalog  
**Depends:** `02_silver_transformation` → `energy_project.silver.*`

---

## 🎯 Goal

Aggregate Silver tables into a **single feature matrix per distributor** — ready for anomaly detection with PCA + Isolation Forest.

Each row in the output represents **one distributor**, with 12 engineered features capturing:
- Consumption behavior (volume, volatility, trend)
- Consumer mix (residential, industrial, commercial)
- Default risk indicators (aging, trend)
- Combined signal (default/consumption ratio)

---

## 📐 Feature Matrix (12 features per distributor)

| Feature | Source | Description |
|---|---|---|
| `avg_consumption_mwh` | silver.samp | Average monthly consumption in MWh |
| `std_consumption_mwh` | silver.samp | Standard deviation of monthly consumption |
| `cv_consumption` | silver.samp | Coefficient of variation (std/avg) — instability signal |
| `consumption_trend` | silver.samp | Slope of consumption over time (growing or declining) |
| `pct_residential` | silver.samp | % of consumption from residential class |
| `pct_industrial` | silver.samp | % of consumption from industrial class |
| `pct_commercial` | silver.samp | % of consumption from commercial class |
| `avg_default_rate` | silver.inadimplencia | Average ITotCrt (total default rate) |
| `avg_aging_12` | silver.inadimplencia | Average ITot12 (12-month aging indicator) |
| `avg_aging_24` | silver.inadimplencia | Average ITot24 (24-month aging indicator) |
| `default_trend` | silver.inadimplencia | Slope of default rate over time |
| `default_consumption_ratio` | both | Default rate / avg consumption — combined risk signal |

---

## 🔄 Pipeline

**silver.samp + silver.inadimplencia → gold.features_distributor → 04_anomaly_model**

---

## 🏛️ Medallion Architecture Context

- **Silver** → Cleansed, typed, deduplicated data
- **Gold** (this layer) → Aggregated, feature-engineered, ML-ready
- **ML** → PCA + Isolation Forest on this feature matrix


In [0]:
# =============================================================================
# LIBRARY IMPORTS
# =============================================================================
# PySpark — distributed processing and aggregations
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window

# NumPy — slope calculation for time series trends
import numpy as np

# Pandas — bridge between Spark and sklearn (sklearn runs on single node)
import pandas as pd

# Scikit-learn — ML pipeline: scaling → PCA → Isolation Forest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest

# MLflow — experiment tracking and model registry
import mlflow
import mlflow.sklearn

print("✅ All libraries imported — ready for Gold feature engineering!")

✅ All libraries imported — ready for Gold feature engineering!


In [0]:
# =============================================================================
# PROJECT CONFIGURATION
# =============================================================================

CATALOG       = "energy_project"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD   = "gold"

# Source tables (Silver)
TABLE_S_SAMP    = f"{CATALOG}.{SCHEMA_SILVER}.samp"
TABLE_S_INADIMP = f"{CATALOG}.{SCHEMA_SILVER}.inadimplencia"

# Destination table (Gold)
TABLE_G_FEATURES = f"{CATALOG}.{SCHEMA_GOLD}.features_distributor"

# Period filter — using full SAMP coverage
YEAR_START = 2020
YEAR_END   = 2026

print("📂 Silver sources:")
print(f"   {TABLE_S_SAMP}")
print(f"   {TABLE_S_INADIMP}")
print()
print("🎯 Gold destination:")
print(f"   {TABLE_G_FEATURES}")
print()
print(f"📅 Period: {YEAR_START} → {YEAR_END}")

📂 Silver sources:
   energy_project.silver.samp
   energy_project.silver.inadimplencia

🎯 Gold destination:
   energy_project.gold.features_distributor

📅 Period: 2020 → 2026


In [0]:
# =============================================================================
# SECTION 1 — READ SILVER SAMP + INSPECT market_detail VALUES
# =============================================================================
# Before aggregating, we need to confirm the exact string values of
# market_detail that represent MWh consumption.
# VlrMercado is polymorphic — same column means different things
# depending on market_detail (MWh, consumers, R$, etc.)

df_samp = spark.table(TABLE_S_SAMP)

print(f"📋 silver.samp — {df_samp.count():,} rows")
print(f"\n🔍 Distinct values of market_detail:")
df_samp.select("market_detail").distinct().orderBy("market_detail").show(50, truncate=False)

📋 silver.samp — 6,319,615 rows

🔍 Distinct values of market_detail:
+----------------------------------+
|market_detail                     |
+----------------------------------+
|COFINS (R$)                       |
|Demanda Contratada kW             |
|Demanda Faturada (kW)             |
|Desconto Demanda %                |
|Desconto Energia TE %             |
|Desconto Energia TUSD %           |
|Desconto de energia compensada %  |
|Energia Consumida (kWh)           |
|Energia Injetada (kWh)            |
|Energia TE (kWh)                  |
|Energia TUSD (kWh)                |
|Energia compensada (kWh)          |
|Energia debitada do SCEE (kWh)    |
|ICMS (R$)                         |
|Nº dias utilizados                |
|Número de Consumidores            |
|Número de consumidores            |
|PIS/COFINS (R$)                   |
|PIS/PASEP (R$)                    |
|PROINFA (kWh)                     |
|Receita Bandeiras (R$)            |
|Receita Demanda (R$)              |
|Receit

In [0]:
# =============================================================================
# SECTION 1 — SAMP FEATURES: CONSUMPTION AGGREGATION
# =============================================================================
# Filter only "Energia Consumida (kWh)" rows — the actual billed consumption.
# Aggregate by distributor: avg, std, and coefficient of variation.
#
# cv_consumption = std / avg → measures instability of consumption over time.
# High cv = erratic consumption pattern → anomaly signal.

df_consumption = (
    df_samp
    .filter(F.col("market_detail") == "Energia Consumida (kWh)")
    .groupBy("distributor_cnpj", "distributor_code", "distributor_name")
    .agg(
        F.avg("market_value").alias("avg_consumption_kwh"),
        F.stddev("market_value").alias("std_consumption_kwh"),
        F.count("market_value").alias("n_months"),
    )
    .withColumn("cv_consumption",
        F.col("std_consumption_kwh") / F.col("avg_consumption_kwh"))
)

print(f"✅ Consumption features computed (lazy)")
print(f"\n🔍 Sample:")
df_consumption.show(5, truncate=False)

✅ Consumption features computed (lazy)

🔍 Sample:
+----------------+----------------+-------------------------------------------------------------------------------+-------------------+-------------------+--------+------------------+
|distributor_cnpj|distributor_code|distributor_name                                                               |avg_consumption_kwh|std_consumption_kwh|n_months|cv_consumption    |
+----------------+----------------+-------------------------------------------------------------------------------+-------------------+-------------------+--------+------------------+
|10835932000108  |Neoenergia PE   |COMPANHIA ENERGETICA DE PERNAMBUCO                                             |1796170.2634859746 |7389182.298898111  |4171    |4.113854042187136 |
|86448057000173  |COORSEL         |COOPERATIVA REGIONAL SUL DE ELETRIFICACAO RURAL                                |31008.28752107926  |74584.88881979192  |1186    |2.4053211183974454|
|87776043000141  |CELETRO     

In [0]:
# =============================================================================
# SECTION 1 — SAMP FEATURES: CONSUMPTION TREND
# =============================================================================
# Calculates the slope of consumption over time per distributor.
# A negative slope = consumption declining → potential fraud signal.
# A very steep positive slope = unusual growth → also worth investigating.
#
# Strategy: convert to Pandas per distributor and use numpy polyfit(deg=1)
# which returns the slope of the best-fit line through the time series.
# We use month index (0,1,2,...) as x and market_value as y.

df_consumption_monthly = (
    df_samp
    .filter(F.col("market_detail") == "Energia Consumida (kWh)")
    .select("distributor_cnpj", "reference_date", "market_value")
    .groupBy("distributor_cnpj", "reference_date")
    .agg(F.sum("market_value").alias("total_kwh"))
    .orderBy("distributor_cnpj", "reference_date")
)

# Convert to Pandas — numpy polyfit runs on single node
pdf_monthly = df_consumption_monthly.toPandas()

# Calculate slope per distributor
slopes = (
    pdf_monthly
    .sort_values(["distributor_cnpj", "reference_date"])
    .groupby("distributor_cnpj")
    .apply(lambda g: np.polyfit(range(len(g)), g["total_kwh"], 1)[0])
    .reset_index()
    .rename(columns={0: "consumption_trend"})
)

print(f"✅ Consumption trend calculated for {len(slopes)} distributors")
print(f"\n🔍 Sample:")
print(slopes.head(5))

/home/spark-2cee5867-fd88-4af9-a89f-8e/.ipykernel/2080/command-6410899339228953-3418669743:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: np.polyfit(range(len(g)), g["total_kwh"], 1)[0])


✅ Consumption trend calculated for 105 distributors

🔍 Sample:
  distributor_cnpj  consumption_trend
0   01229747000189       1.056014e+04
1   01377555000110       4.539650e+04
2   01543032000104       4.053170e+06
3   02016440000162       3.876138e+06
4   02302100000106       8.208483e+05


In [0]:
# =============================================================================
# SECTION 1 — SAMP FEATURES: CONSUMER MIX
# =============================================================================
# Calculates the share of consumption per class for each distributor.
# Mix imbalances can signal anomalies — e.g. a "residential" distributor
# suddenly showing heavy industrial consumption.
#
# Fix applied: include_groups=False in groupby().apply() to silence
# the pandas DeprecationWarning from the previous cell.

# ── Fix slopes calculation (suppress DeprecationWarning) ─────────────────────
slopes = (
    pdf_monthly
    .sort_values(["distributor_cnpj", "reference_date"])
    .groupby("distributor_cnpj")[["total_kwh"]]
    .apply(lambda g: np.polyfit(range(len(g)), g["total_kwh"], 1)[0],
           include_groups=False)
    .reset_index()
    .rename(columns={0: "consumption_trend"})
)

# ── Consumer mix ──────────────────────────────────────────────────────────────
df_mix = (
    df_samp
    .filter(F.col("market_detail") == "Energia Consumida (kWh)")
    .groupBy("distributor_cnpj")
    .agg(
        F.sum(F.when(F.col("consumption_class") == "Residencial",
              F.col("market_value"))).alias("kwh_residential"),
        F.sum(F.when(F.col("consumption_class") == "Industrial",
              F.col("market_value"))).alias("kwh_industrial"),
        F.sum(F.when(F.col("consumption_class") == "Comercial",
              F.col("market_value"))).alias("kwh_commercial"),
        F.sum("market_value").alias("kwh_total"),
    )
    .withColumn("pct_residential", F.col("kwh_residential") / F.col("kwh_total"))
    .withColumn("pct_industrial",  F.col("kwh_industrial")  / F.col("kwh_total"))
    .withColumn("pct_commercial",  F.col("kwh_commercial")  / F.col("kwh_total"))
    .select("distributor_cnpj", "pct_residential", "pct_industrial", "pct_commercial")
)

print(f"✅ Consumer mix features computed (lazy)")
print(f"\n🔍 Sample:")
df_mix.show(5, truncate=False)

✅ Consumer mix features computed (lazy)

🔍 Sample:
+----------------+--------------------+-------------------+--------------------+
|distributor_cnpj|pct_residential     |pct_industrial     |pct_commercial      |
+----------------+--------------------+-------------------+--------------------+
|01543032000104  |0.4071271434342514  |0.05134639098641354|0.33351851537678473 |
|09095183000140  |0.3663854098650841  |0.09003855698878703|0.44986473931732407 |
|11810343000138  |0.28725965081205357 |0.183185573099013  |0.5295547760889334  |
|90660754000160  |0.056896770612597035|0.09620163294514543|0.1263801711609782  |
|10532365000110  |0.013390815895172612|0.3426761111474    |0.034528282295465444|
+----------------+--------------------+-------------------+--------------------+
only showing top 5 rows


In [0]:
# =============================================================================
# SECTION 2 — INADIMPLENCIA FEATURES
# =============================================================================

df_inadimp = spark.table(TABLE_S_INADIMP)

# ── Average default indicators ────────────────────────────────────────────────
df_default = (
    df_inadimp
    .filter(F.col("reference_year").between(YEAR_START, YEAR_END))
    .groupBy("distributor_cnpj")
    .agg(
        F.avg(F.when(F.col("indicator_code") == "ITotCrt",
              F.col("indicator_value"))).alias("avg_default_rate"),
        F.avg(F.when(F.col("indicator_code") == "ITot12",
              F.col("indicator_value"))).alias("avg_aging_12"),
        F.avg(F.when(F.col("indicator_code") == "ITot24",
              F.col("indicator_value"))).alias("avg_aging_24"),
    )
)

# ── Default trend (slope of ITotCrt over time) ────────────────────────────────
pdf_default_monthly = (
    df_inadimp
    .filter(
        (F.col("reference_year").between(YEAR_START, YEAR_END)) &
        (F.col("indicator_code") == "ITotCrt")
    )
    .groupBy("distributor_cnpj", "reference_date")
    .agg(F.avg("indicator_value").alias("default_rate"))
    .orderBy("distributor_cnpj", "reference_date")
    .toPandas()
)

# Safe slope — returns 0.0 if polyfit fails (too few points or no variation)
def safe_slope(g):
    try:
        return np.polyfit(range(len(g)), g["default_rate"], 1)[0]
    except Exception:
        return 0.0

default_slopes = (
    pdf_default_monthly
    .sort_values(["distributor_cnpj", "reference_date"])
    .groupby("distributor_cnpj")[["default_rate"]]
    .apply(safe_slope, include_groups=False)
    .reset_index()
    .rename(columns={0: "default_trend"})
)

print(f"✅ Default features computed")
print(f"   Distributors with default data: {len(default_slopes)}")
print(f"\n🔍 Sample df_default:")
df_default.show(5, truncate=False)
print(f"\n🔍 Sample default_slopes:")
print(default_slopes.head(5))

/databricks/python/lib/python3.12/site-packages/numpy/lib/_polynomial_impl.py:665: RuntimeWarning: invalid value encountered in divide
  lhs /= scale


✅ Default features computed
   Distributors with default data: 105

🔍 Sample df_default:
+----------------+------------------+-------------------+-------------------+
|distributor_cnpj|avg_default_rate  |avg_aging_12       |avg_aging_24       |
+----------------+------------------+-------------------+-------------------+
|09257558000121  |327.0405405405405 |0.2039189189189189 |0.16081081081081078|
|86532348000145  |76.46575342465754 |0.1419178082191781 |0.08575342465753427|
|01229747000189  |11.346666666666666|0.9610666666666666 |0.9246666666666666 |
|08826596000195  |5033.4390243902435|0.6290243902439024 |0.4878048780487806 |
|86533346000170  |168.16326530612244|0.37816326530612254|0.1489795918367347 |
+----------------+------------------+-------------------+-------------------+
only showing top 5 rows

🔍 Sample default_slopes:
  distributor_cnpj  default_trend
0   01229747000189       0.209331
1   01377555000110       4.224953
2   01543032000104     600.343961
3   02016440000162     

In [0]:
# =============================================================================
# SECTION 3 — JOIN ALL FEATURES INTO SINGLE MATRIX
# =============================================================================
# Joins all feature DataFrames into one row per distributor.
# Also computes the combined feature:
#   default_consumption_ratio = avg_default_rate / avg_consumption_kwh
#
# Uses left join on distributor_cnpj — distributors without default data
# will have nulls, filled with 0 (no reported default = low risk signal).

# Convert slopes (pandas) back to Spark for the join
df_consumption_trend = spark.createDataFrame(slopes)
df_default_trend     = spark.createDataFrame(default_slopes)

# ── Join all features ─────────────────────────────────────────────────────────
df_features = (
    df_consumption
    .join(df_consumption_trend, on="distributor_cnpj", how="left")
    .join(df_mix,               on="distributor_cnpj", how="left")
    .join(df_default,           on="distributor_cnpj", how="left")
    .join(df_default_trend,     on="distributor_cnpj", how="left")
    # Combined feature
    .withColumn("default_consumption_ratio",
        F.col("avg_default_rate") / F.col("avg_consumption_kwh"))
    # Fill nulls with 0 (no default data = no reported default)
    .fillna(0.0)
    # Keep only the 12 model features + identifier columns
    .select(
        "distributor_cnpj",
        "distributor_code",
        "distributor_name",
        "avg_consumption_kwh",
        "std_consumption_kwh",
        "cv_consumption",
        "consumption_trend",
        "pct_residential",
        "pct_industrial",
        "pct_commercial",
        "avg_default_rate",
        "avg_aging_12",
        "avg_aging_24",
        "default_trend",
        "default_consumption_ratio",
    )
)

print(f"✅ Feature matrix assembled (lazy)")
print(f"\n📋 Schema:")
df_features.printSchema()
print(f"\n🔍 Sample:")
df_features.show(3, truncate=False)

✅ Feature matrix assembled (lazy)

📋 Schema:
root
 |-- distributor_cnpj: string (nullable = true)
 |-- distributor_code: string (nullable = true)
 |-- distributor_name: string (nullable = true)
 |-- avg_consumption_kwh: double (nullable = false)
 |-- std_consumption_kwh: double (nullable = false)
 |-- cv_consumption: double (nullable = false)
 |-- consumption_trend: double (nullable = false)
 |-- pct_residential: double (nullable = false)
 |-- pct_industrial: double (nullable = false)
 |-- pct_commercial: double (nullable = false)
 |-- avg_default_rate: double (nullable = false)
 |-- avg_aging_12: double (nullable = false)
 |-- avg_aging_24: double (nullable = false)
 |-- default_trend: double (nullable = false)
 |-- default_consumption_ratio: double (nullable = false)


🔍 Sample:
+----------------+----------------+---------------------------------------------------------------------+-------------------+-------------------+------------------+------------------+-------------------+-----

In [0]:
# =============================================================================
# SECTION 4 — WRITE TO GOLD
# =============================================================================
# Single action of this notebook — triggers the full DAG.
# No partitioning needed: ~105 rows is too small to benefit from it.

print(f"💾 Writing to {TABLE_G_FEATURES} ...")

(
    df_features.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_G_FEATURES)
)

print(f"✅ gold.features_distributor written successfully!")

# ── Validation ────────────────────────────────────────────────────────────────
df_gold = spark.table(TABLE_G_FEATURES)
count   = df_gold.count()

print(f"\n{'='*60}")
print(f"🔍 VALIDATION — gold.features_distributor")
print(f"{'='*60}")
print(f"\n📊 Distributors: {count}")
print(f"\n❓ Null check:")
df_gold.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in df_gold.columns if c not in
    ["distributor_cnpj", "distributor_code", "distributor_name"]
]).show(truncate=False)
print(f"\n📊 Feature statistics:")
df_gold.select(
    "avg_consumption_kwh", "cv_consumption",
    "avg_default_rate", "default_consumption_ratio"
).describe().show(truncate=False)
print(f"{'='*60}")
print(f"✅ Gold layer complete — ready for 04_anomaly_model!")
print(f"{'='*60}")

💾 Writing to energy_project.gold.features_distributor ...
✅ gold.features_distributor written successfully!

🔍 VALIDATION — gold.features_distributor

📊 Distributors: 105

❓ Null check:
+-------------------+-------------------+--------------+-----------------+---------------+--------------+--------------+----------------+------------+------------+-------------+-------------------------+
|avg_consumption_kwh|std_consumption_kwh|cv_consumption|consumption_trend|pct_residential|pct_industrial|pct_commercial|avg_default_rate|avg_aging_12|avg_aging_24|default_trend|default_consumption_ratio|
+-------------------+-------------------+--------------+-----------------+---------------+--------------+--------------+----------------+------------+------------+-------------+-------------------------+
|0                  |0                  |0             |0                |0              |0             |0             |0               |0           |0           |0            |0                        